# Aegis — Real-Model Evaluation (Path B: Colab)

This notebook runs the Aegis real-model evaluation entirely inside a free Colab VM,
using a local `llama.cpp` OpenAI-compatible server. No API token or tunnel is needed.

**What it does, step by step:**
1. Install Node 20 (the eval runner is TypeScript) and clone the repo.
2. Install `llama-cpp-python` and download a 4-bit GGUF model (fits a free T4's 16 GB).
3. Launch an OpenAI-compatible server on `127.0.0.1:8000` and health-check it.
4. Run `npm run llm-eval` against that local server.
5. Show the summary table.
6. (Optional) Copy the raw report to Google Drive so it survives the session.

> **Runtime:** set **Runtime → Change runtime type → T4 GPU** before running.
> A full single-model sweep takes roughly 30–45 minutes on the free tier.

## 1. Install Node 20 and clone the repo

In [ ]:
# Install Node 20 via NodeSource, then clone the repository.
# Edit BRANCH if the eval code is on a feature branch rather than the default.
REPO_URL = 'https://github.com/holeyfield33-art/aegis-provenance.git'
BRANCH = 'main'

!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y nodejs > /dev/null 2>&1
!node --version

import os
if not os.path.isdir('aegis-provenance'):
    !git clone --branch $BRANCH --depth 1 $REPO_URL
%cd aegis-provenance
!git rev-parse --short HEAD

## 2. Install the local model server and download a GGUF model

We use `llama-cpp-python`'s OpenAI-compatible server and a 4-bit quantized
`Qwen2.5-7B-Instruct` (Q4_K_M ≈ 4.7 GB). Any GGUF chat model works — change
`GGUF_REPO` / `GGUF_FILE` to try another.

In [ ]:
!pip install -q 'llama-cpp-python[server]' huggingface_hub

from huggingface_hub import hf_hub_download

GGUF_REPO = 'Qwen/Qwen2.5-7B-Instruct-GGUF'
GGUF_FILE = 'qwen2.5-7b-instruct-q4_k_m.gguf'
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'  # label used in the eval report

model_path = hf_hub_download(repo_id=GGUF_REPO, filename=GGUF_FILE)
print('Downloaded to:', model_path)

## 3. Launch the OpenAI-compatible server and health-check it

The server runs in the background on port 8000. We poll `/v1/models` until it is ready.
`--n_gpu_layers -1` offloads all layers to the T4 GPU.

In [ ]:
import subprocess, time, urllib.request, json

server = subprocess.Popen([
    'python', '-m', 'llama_cpp.server',
    '--model', model_path,
    '--n_gpu_layers', '-1',
    '--n_ctx', '4096',
    '--host', '127.0.0.1',
    '--port', '8000',
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def ready():
    try:
        with urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

for _ in range(120):
    if ready():
        print('Server is up.')
        break
    time.sleep(2)
else:
    raise RuntimeError('Server did not start in time — check the GPU runtime is enabled.')

## 4. Run the evaluation

`npm ci` installs the repo's dependencies, then `npm run llm-eval` drives the local
server. `AEGIS_EVAL_API_KEY` can be any non-empty string — the local server ignores it.

In [ ]:
import os
os.environ['AEGIS_EVAL_BASE_URL'] = 'http://127.0.0.1:8000/v1'
os.environ['AEGIS_EVAL_API_KEY'] = 'local-dummy-key'
os.environ['AEGIS_EVAL_MODELS'] = MODEL_ID
os.environ['AEGIS_EVAL_CONCURRENCY'] = '1'  # one local GPU: keep it serial

!npm ci
!npm run llm-eval

## 5. Display the summary table

In [ ]:
from IPython.display import Markdown, display
with open('llm-eval-summary.md') as f:
    display(Markdown(f.read()))

## 6. (Optional) Save the raw report to Google Drive

Colab sessions are ephemeral. Run this to copy `llm-eval-report.json` to your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/aegis-eval
!cp llm-eval-report.json llm-eval-summary.md /content/drive/MyDrive/aegis-eval/
print('Saved to Drive: MyDrive/aegis-eval/')